# Generational Differences in Sharing Visuals by Phone and Preserving Digital Files

The term **visuals** refers to photos and videos. This Notebook calculates results from a SAV you upload, then provides nine figures and a downloadable results ZIP.

**Start here:** upload one SAV to `data/official`, then choose **Run → Run All Cells**. The default settings discover that file and use its metadata to prepare the analysis.

Obtain official data through the [provider's access channels](../docs/DATA_ACCESS.md). Use Binder only when you are permitted to process those data on this cloud service; otherwise use the same Notebook locally. The repository contains no respondent records.

## 1. Upload your SAV

In JupyterLab's left file browser, click the **home icon**, open **data → official**, and click **Upload Files**. Upload one `.sav` file; its original filename is fine. Then return to this Notebook and select **Run → Run All Cells**.

The upload folder on Binder is `/home/jovyan/data/official/`. On a local checkout, it is the `data/official/` folder inside the repository; create it if needed. Uploads are temporary on Binder and are not committed to GitHub or included in the results ZIP.

**No data yet?** Set `MODE = "aggregate"` below to preview the supplied results. This redraws existing statistical tables rather than recalculating them from respondent records.

## 2. Settings

For one uploaded SAV with unambiguous variable and response labels, leave these settings unchanged. If setup needs clarification, the next cell stops and explains which inputs need confirmation. Follow the [input-profile guide](../docs/INPUT_PROFILE.md); a confirmed `profile.json` can be reused beside the SAV.

Optional examples:

```python
DATA_FILE = "/home/jovyan/data/official/survey.sav"  # Select one file if you uploaded several.
OUTPUT_DIR = "/tmp/tscs-results/run-01"              # Binder; choose a new folder for each run.
# Local example: OUTPUT_DIR = "~/Documents/TSCS/results/run-01"
```

`CALIBRATION_FILE` supplies optional population targets for the additional recalibration check. Leave it blank when targets are unavailable. `FONT` can stay blank on Binder. After changing settings, select **Run → Run All Cells** again.

In [ ]:
MODE = "survey"       # "survey": calculate from your SAV; "aggregate": preview supplied results.
DATA_FILE = ""         # Blank finds the single SAV in data/official (any filename).
PROFILE_FILE = ""      # Optional confirmed profile; blank uses profile.json or SAV metadata.
CALIBRATION_FILE = ""  # Optional population-target JSON; blank skips that check.
OUTPUT_DIR = ""        # Blank creates a new temporary result folder on each run.
FONT = ""              # Blank uses the installed default.
INPUT_OVERRIDES = {}    # Advanced: confirmed columns/value_maps/design; see the input-profile guide.


## 3. Check the input setup

The next cell identifies the uploaded file and prepares a local mapping from its metadata. It does not guess ambiguous weights, response categories or sampling units. If a required input is unclear, analysis stops here with instructions for completing the setup.

The selected paths and setup summary appear below. Review any warning before interpreting results. The [Notebook guide](../docs/NOTEBOOK.md) covers local execution and troubleshooting.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "run.py").is_file() and (candidate / "manifest.json").is_file():
        sys.path.insert(0, str(candidate))
        break

from helpers.notebook_workflow import configuration, find_root, run_workflow
from helpers.survey_setup import prepare_upload

root = find_root()
settings = configuration({
    "REPRO_MODE": MODE,
    "REPRO_DATA_FILE": DATA_FILE,
    "REPRO_PROFILE_FILE": PROFILE_FILE,
    "REPRO_CALIBRATION_FILE": CALIBRATION_FILE,
    "REPRO_OUTPUT_DIR": OUTPUT_DIR,
    "REPRO_FONT": FONT,
})
print("Mode:", settings.mode)
if settings.mode == "survey":
    prepared = prepare_upload(root, settings.data_file, settings.profile_file, INPUT_OVERRIDES)
    settings = replace(settings, data_file=prepared.data_file, profile_file=prepared.profile_file)
    print(prepared.summary)
    print("Survey file:", settings.data_file)
    print("Mapping profile:", settings.profile_file)
    print("Population targets:", settings.calibration_file or "Not supplied")
else:
    print("Input: published aggregate tables included in this repository")
print("Output directory:", settings.output_dir)


## 4. Run the analysis and draw the figures

In `aggregate` mode, this step verifies the supplied tables, multiple-testing adjustments and sensitivity summaries.

In `survey` mode, it reads your SAV, applies the mapping profile, estimates the group results and runs the specified statistical comparisons and sensitivity checks. This can take several minutes. Missing required inputs stop the run; stored results are never substituted for a missing calculation.

Both modes produce nine figures and links to a preview and results ZIP.

In [ ]:
report = run_workflow(root, settings)
if report["statistical_reestimation"]:
    print("Calculated results from the selected survey file.")
    if report.get("artificial_data"):
        print("ARTIFICIAL DEMONSTRATION DATA: these are not empirical study results.")
    if not report["calibration_available"]:
        print("Additional population recalibration was not run: no targets were supplied.")
else:
    print("Verified published aggregates; survey statistics were not reestimated.")
print("Research figures:", report["figure_count"])

import json
from helpers.notebook_exports import export_results
from helpers.notebook_display import display_results

exports = export_results(root, settings.output_dir.resolve(), root / "notebooks/downloads")
download_report = {key: str(exports[key]) for key in ("directory", "index_path", "zip_path")}
(settings.output_dir / "download_report.json").write_text(
    json.dumps(download_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Results directory:", settings.output_dir)
display_results(exports)

## 5. Inspect the figures

The figures compare weighted proportions within age groups, education groups, and groups within digitally mature or non-mature areas. The source mode is shown below. The plotting rows are saved in `figures/selected_profiles.csv`.

See the [terminology guide](../docs/TERMINOLOGY.md) for activity definitions, denominators and storage categories.

In [ ]:
from IPython.display import Image, Markdown, display

source = ("Reestimated from the selected survey file" if report["statistical_reestimation"]
          else "Published aggregates; not reestimated")
if report.get("artificial_data"):
    source = "Artificial demonstration data; not empirical study results"
display(Markdown(f"**Figure data source: {source}.**"))
for relative in report["figure_files"]:
    display(Image(filename=str(settings.output_dir / relative), width=1000))

## 6. Download the results

Use **View nine figures** or **Download figures and aggregates (ZIP)** above. The ZIP contains PNG/PDF figures, plotting aggregates, ten statistical result tables, and reading information. In `survey` mode, these are the newly calculated tables. It excludes the SAV, mapping profile, respondent records and execution logs.

Download copies are created under `notebooks/downloads/`, where Jupyter serves the links; they are ignored by Git. After extracting the ZIP, open `index.html` to view the figures. The tables are in `tables/primary/`, `tables/alternative/` and `tables/contrasts/`.

Binder has temporary storage. Download the results before ending the session. Clear execution outputs and reset personal paths before contributing a Notebook change.